# Robust Electricity Theft Detection Under Data-Poisoning Attacks
## Complete Experiment Pipeline — Colab A100
### Shallow ML (Tuned) + Deep Learning (Tuned) + Ensemble + All Metrics + All Plots

This notebook runs end-to-end. Every cell depends only on cells above it.
SMOTE is handled correctly — inside CV folds for tuning, applied once for final training.
No data leakage. Index-verified splits. Checkpointed to Google Drive.

## 1. Setup

In [ ]:
!pip install imbalanced-learn xgboost lightgbm keras-tuner -q

from google.colab import drive
drive.mount('/content/drive')

import os, json, time, warnings, joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split, RandomizedSearchCV, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.metrics import (
    f1_score, recall_score, precision_score, accuracy_score,
    roc_auc_score, matthews_corrcoef, confusion_matrix,
    roc_curve, ConfusionMatrixDisplay, make_scorer
)
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

SAVE_DIR = '/content/drive/MyDrive/sgcc_final_results'
os.makedirs(SAVE_DIR, exist_ok=True)

POISON_RATES = [0.0, 0.10, 0.20, 0.30]

gpus = tf.config.list_physical_devices('GPU')
print(f"TF: {tf.__version__}, GPU: {gpus}")
print(f"Save dir: {SAVE_DIR}")
print("setup done")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.4/129.4 kB 3.6 MB/s eta 0:00:00
Mounted at /content/drive
TF: 2.20.0, GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Save dir: /content/drive/MyDrive/sgcc_final_results
setup done


## 2. Load Data and Preprocess

In [ ]:
# Update this path if your CSV is in a different Drive location
df_raw = pd.read_csv('/content/drive/MyDrive/data/data.csv')

LABEL_COL = 'FLAG'
y_all = df_raw[LABEL_COL].values
features_raw = df_raw.select_dtypes(include=['number'])
if LABEL_COL in features_raw.columns:
    features_raw = features_raw.drop(columns=[LABEL_COL])

# Impute -> Scale -> PCA
imputer = SimpleImputer(strategy='median')
X_imputed = imputer.fit_transform(features_raw)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_imputed)

pca = PCA(n_components=0.95, random_state=SEED)
X_pca = pca.fit_transform(X_scaled)

print(f"Raw: {features_raw.shape} -> PCA: {X_pca.shape} ({pca.n_components_} components)")

Raw: (42372, 1034) -> PCA: (42372, 42) (42 components)


## 3. Train/Val/Test Split — Index-Based, Leakage-Verified

In [ ]:
# Split by index to guarantee no row-level confusion
all_idx = np.arange(len(X_pca))

train_idx, temp_idx = train_test_split(
    all_idx, test_size=0.30, stratify=y_all, random_state=SEED)
val_idx, test_idx = train_test_split(
    temp_idx, test_size=0.50, stratify=y_all[temp_idx], random_state=SEED)

# Verify zero index overlap
assert len(set(train_idx) & set(test_idx)) == 0, "train/test overlap"
assert len(set(train_idx) & set(val_idx)) == 0, "train/val overlap"
assert len(set(val_idx) & set(test_idx)) == 0, "val/test overlap"

X_train_raw, y_train_raw = X_pca[train_idx], y_all[train_idx]
X_val, y_val = X_pca[val_idx], y_all[val_idx]
X_test, y_test = X_pca[test_idx], y_all[test_idx]

print(f"Train: {X_train_raw.shape} (Normal:{(y_train_raw==0).sum()}, Theft:{(y_train_raw==1).sum()})")
print(f"Val:   {X_val.shape}")
print(f"Test:  {X_test.shape} (Normal:{(y_test==0).sum()}, Theft:{(y_test==1).sum()})")
print("Index overlap: 0/0/0 — verified clean")

Train: (29660, 42) (Normal:27130, Theft:2530)
Val:   (6356, 42)
Test:  (6356, 42) (Normal:5814, Theft:542)
Index overlap: 0/0/0 — verified clean


## 4. Poison Injection

In [ ]:
def inject_poison(y_labels, poison_rate, seed=SEED):
    rng = np.random.default_rng(seed)
    y_poisoned = y_labels.copy()
    n_poison = int(len(y_labels) * poison_rate)
    flipped_idx = rng.choice(len(y_labels), size=n_poison, replace=False)
    y_poisoned[flipped_idx] = 1 - y_poisoned[flipped_idx]
    return y_poisoned

# Generate poisoned versions of the RAW (pre-SMOTE) training labels
poisoned_labels_raw = {}
for rate in POISON_RATES:
    poisoned_labels_raw[rate] = inject_poison(y_train_raw, rate)
    flipped = (poisoned_labels_raw[rate] != y_train_raw).sum()
    print(f"  {int(rate*100):>2}% poison: {flipped} labels flipped")

   0% poison: 0 labels flipped
  10% poison: 2966 labels flipped
  20% poison: 5932 labels flipped
  30% poison: 8898 labels flipped


## 5. Evaluation Function

In [ ]:
def evaluate(y_true, y_pred, y_proba, model_name, poison_rate, train_time=0):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return {
        'Model': model_name,
        'Poison_Rate': f'{int(poison_rate*100)}%',
        'Poison_Float': poison_rate,
        'F1': round(f1_score(y_true, y_pred), 4),
        'DR': round(recall_score(y_true, y_pred), 4),
        'Precision': round(precision_score(y_true, y_pred, zero_division=0), 4),
        'Accuracy': round(accuracy_score(y_true, y_pred), 4),
        'AUC': round(roc_auc_score(y_true, y_proba), 4),
        'FPR': round(fp / (fp + tn), 4),
        'MCC': round(matthews_corrcoef(y_true, y_pred), 4),
        'Train_Time_s': round(train_time, 2),
        'y_proba': y_proba,
        'y_pred': y_pred,
    }

print("evaluate() ready — reports F1, DR, Precision, Accuracy, AUC, FPR, MCC")

evaluate() ready — reports F1, DR, Precision, Accuracy, AUC, FPR, MCC


In [ ]:
from sklearn.metrics import f1_score, recall_score, precision_score, accuracy_score, roc_auc_score, matthews_corrcoef, confusion_matrix

def evaluate(y_true, y_pred, y_proba, model_name, poison_rate, train_time=0):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return {
        'Model': model_name,
        'Poison_Rate': f'{int(poison_rate*100)}%',
        'Poison_Float': poison_rate,
        'F1': round(f1_score(y_true, y_pred), 4),
        'DR': round(recall_score(y_true, y_pred), 4),
        'Precision': round(precision_score(y_true, y_pred, zero_division=0), 4),
        'Accuracy': round(accuracy_score(y_true, y_pred), 4),
        'AUC': round(roc_auc_score(y_true, y_proba), 4),
        'FPR': round(fp / (fp + tn), 4),
        'MCC': round(matthews_corrcoef(y_true, y_pred), 4),
        'Train_Time_s': round(train_time, 2),
        'y_proba': y_proba,
        'y_pred': y_pred,
    }

cols = ['Model','Poison_Rate','F1','DR','Precision','Accuracy','AUC','FPR','MCC','Train_Time_s']
print("evaluate() redefined")

evaluate() redefined


## 6. Shallow ML — Hyperparameter Tuning (SMOTE Inside CV Folds)

SMOTE is applied inside each CV fold via imblearn Pipeline.
This prevents synthetic samples from leaking across fold boundaries.
Same N_ITER=20 and cv=5 for every model — fair, comparable tuning budget.

In [ ]:
N_ITER = 20
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
scorer = make_scorer(f1_score)

pos_weight = (y_train_raw==0).sum() / (y_train_raw==1).sum()

def make_pipe(clf):
    return ImbPipeline([('smote', SMOTE(random_state=SEED)), ('clf', clf)])

search_spaces = {
    'RandomForest': (make_pipe(RandomForestClassifier(random_state=SEED, n_jobs=1)), {
        'clf__n_estimators': [100, 200, 300, 500],
        'clf__max_depth': [None, 5, 10, 15, 20],
        'clf__min_samples_split': [2, 5, 10],
        'clf__class_weight': ['balanced', None],
    }),
    'AdaBoost': (make_pipe(AdaBoostClassifier(random_state=SEED)), {
        'clf__n_estimators': [50, 100, 200, 300],
        'clf__learning_rate': [0.01, 0.05, 0.1, 0.5, 1.0],
    })

}

best_shallow_params = {}
shallow_tuning_log = []

for name, (pipeline, params) in search_spaces.items():
    print(f'\nTuning {name}...')
    search = RandomizedSearchCV(
        pipeline, params, n_iter=N_ITER, scoring=scorer,
        cv=cv, random_state=SEED, n_jobs=-1, verbose=1
    )
    search.fit(X_train_raw, y_train_raw)
    best_shallow_params[name] = search.best_params_
    shallow_tuning_log.append({
        'model': name, 'best_cv_f1': search.best_score_,
        'best_params': search.best_params_, 'n_configs': N_ITER
    })
    print(f'  Best CV F1: {search.best_score_:.4f}')
    print(f'  Best params: {search.best_params_}')

with open(f'{SAVE_DIR}/shallow_tuning_log.json', 'w') as f:
    json.dump(shallow_tuning_log, f, indent=2, default=str)
print("\nShallow tuning complete — log saved to Drive")


Tuning RandomForest...
Fitting 5 folds for each of 20 candidates, totalling 100 fits
  Best CV F1: 0.3302
  Best params: {'clf__n_estimators': 500, 'clf__min_samples_split': 10, 'clf__max_depth': None, 'clf__class_weight': 'balanced'}

Tuning AdaBoost...
Fitting 5 folds for each of 20 candidates, totalling 100 fits


KeyboardInterrupt: 

In [ ]:
smote = SMOTE(random_state=SEED)
X_tr_s, y_tr_s = smote.fit_resample(X_train_raw, y_train_raw)

rf_tuned = RandomForestClassifier(
    n_estimators=500,
    max_depth=None,
    min_samples_split=10,
    class_weight='balanced',
    random_state=SEED,
    n_jobs=-1
)
rf_tuned.fit(X_tr_s, y_tr_s)

proba = rf_tuned.predict_proba(X_test)[:, 1]
pred = (proba >= 0.5).astype(int)

print("Tuned RF — TEST SET (same test set as Week 2):")
print(f"  F1:        {f1_score(y_test, pred):.4f}")
print(f"  Recall:    {recall_score(y_test, pred):.4f}")
print(f"  Precision: {precision_score(y_test, pred):.4f}")
print(f"  AUC:       {roc_auc_score(y_test, proba):.4f}")
print(f"  MCC:       {matthews_corrcoef(y_test, pred):.4f}")
print()
print("Week 2 original RF (untuned) for comparison: F1=0.4562")

Tuned RF — TEST SET (same test set as Week 2):
  F1:        0.3519
  Recall:    0.3616
  Precision: 0.3427
  AUC:       0.7877
  MCC:       0.2898

Week 2 original RF (untuned) for comparison: F1=0.4562


## 7. Train Tuned Shallow Models at All Poison Levels

For each poison level: apply SMOTE to poisoned training labels, train with best params, evaluate on clean test set.

In [ ]:
def extract_clf_params(best_params):
    return {k.replace('clf__', ''): v for k, v in best_params.items() if k.startswith('clf__')}

shallow_configs = {
    'RandomForest': RandomForestClassifier,
    'AdaBoost': AdaBoostClassifier,
    'XGBoost': XGBClassifier,
    'LightGBM': LGBMClassifier,
}

shallow_results = []

for name, ModelClass in shallow_configs.items():
    params = extract_clf_params(best_shallow_params[name])
    print(f'\n{name}')
    print('-' * 40)

    for rate in POISON_RATES:
        y_poisoned = poisoned_labels_raw[rate]

        smote = SMOTE(random_state=SEED)
        X_tr_smote, y_tr_smote = smote.fit_resample(X_train_raw, y_poisoned)

        extra = {'random_state': SEED}
        if name == 'RandomForest':
            extra['n_jobs'] = -1
        if name == 'XGBoost':
            extra.update({'eval_metric': 'logloss', 'tree_method': 'hist', 'device': 'cuda'})
        if name == 'LightGBM':
            extra['verbose'] = -1

        model = ModelClass(**params, **extra)
        t0 = time.time()
        model.fit(X_tr_smote, y_tr_smote)
        elapsed = time.time() - t0

        proba = model.predict_proba(X_test)[:, 1]
        pred = (proba >= 0.5).astype(int)

        result = evaluate(y_test, pred, proba, name, rate, elapsed)
        shallow_results.append(result)

        print(f'  {int(rate*100):>2}% — F1={result["F1"]:.4f}  DR={result["DR"]:.4f}  '
              f'AUC={result["AUC"]:.4f}  MCC={result["MCC"]:.4f}  ({elapsed:.1f}s)')

# Save
cols = ['Model','Poison_Rate','F1','DR','Precision','Accuracy','AUC','FPR','MCC','Train_Time_s']
shallow_df = pd.DataFrame([{k:v for k,v in r.items() if k in cols} for r in shallow_results])
shallow_df.to_csv(f'{SAVE_DIR}/shallow_results.csv', index=False)
print('\nSaved: shallow_results.csv')

## 8. Deep Learning — GRU Hyperparameter Tuning

Matching Takiddin et al. Table II hyperparameter ranges.
Same tuning budget (20 trials) as shallow models for fairness.
SMOTE applied once to training set before tuning (standard for DL — CV-internal SMOTE
is impractical for neural networks due to training cost per fold).

In [ ]:
!pip install keras-tuner -q

In [ ]:
import keras_tuner as kt

# SMOTE the clean training data for DL tuning
smote = SMOTE(random_state=SEED)
X_train_smote, y_train_smote = smote.fit_resample(X_train_raw, y_train_raw)

# Reshape for sequence models
X_tr_seq = X_train_smote.reshape(X_train_smote.shape[0], X_train_smote.shape[1], 1)
X_val_seq = X_val.reshape(X_val.shape[0], X_val.shape[1], 1)
X_test_seq = X_test.reshape(X_test.shape[0], X_test.shape[1], 1)

print(f"DL training data: {X_tr_seq.shape}, Val: {X_val_seq.shape}, Test: {X_test_seq.shape}")

def build_gru(hp):
    model = keras.Sequential()
    model.add(layers.Input(shape=(X_tr_seq.shape[1], 1)))

    n_layers = hp.Choice('n_layers', [2, 4, 6, 8])
    n_cells = hp.Choice('n_cells', [100, 200, 300])
    dropout = hp.Choice('dropout', [0.0, 0.2, 0.4])

    for i in range(n_layers):
        return_seq = (i < n_layers - 1)
        model.add(layers.GRU(n_cells, return_sequences=return_seq))
        if dropout > 0:
            model.add(layers.Dropout(dropout))

    model.add(layers.Dense(1, activation='sigmoid'))

    lr = hp.Choice('learning_rate', [0.001, 0.0005, 0.0001])
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr),
        loss='binary_crossentropy',
        metrics=[keras.metrics.AUC(name='auc')]
    )
    return model

gru_tuner = kt.RandomSearch(
    build_gru,
    objective=kt.Objective('val_auc', direction='max'),
    max_trials=20,
    executions_per_trial=1,
    directory=f'{SAVE_DIR}/gru_tuning',
    project_name='gru',
    seed=SEED
)

gru_tuner.search(
    X_tr_seq, y_train_smote,
    validation_data=(X_val_seq, y_val),
    epochs=30,
    batch_size=256,
    callbacks=[EarlyStopping(patience=5, restore_best_weights=True, verbose=0)],
    verbose=1
)

best_gru_hp = gru_tuner.get_best_hyperparameters(1)[0]
print(f"\nBest GRU config: {best_gru_hp.values}")

# Save best config
with open(f'{SAVE_DIR}/gru_best_hp.json', 'w') as f:
    json.dump(best_gru_hp.values, f, indent=2)
print("GRU tuning complete — saved to Drive")

Trial 20 Complete [00h 01m 15s]
val_auc: 0.678068220615387

Best val_auc So Far: 0.7275221347808838
Total elapsed time: 00h 52m 09s

Best GRU config: {'n_layers': 6, 'n_cells': 300, 'dropout': 0.2, 'learning_rate': 0.001}
GRU tuning complete — saved to Drive


## 8. Deep Learning — GRU Hyperparameter Tuning with exact-match Takiddin architecture

Matching Takiddin et al. Table II hyperparameter ranges.
Same tuning budget (20 trials) as shallow models for fairness.
SMOTE applied once to training set before tuning (standard for DL — CV-internal SMOTE
is impractical for neural networks due to training cost per fold).

In [ ]:
# Use raw scaled sequence, NOT PCA — required for GRU to see real temporal structure
X_train_raw_seq = X_scaled[train_idx]   # X_scaled = post-impute, post-StandardScaler, pre-PCA
y_train_raw = y_all[train_idx]
X_val_seq_full = X_scaled[val_idx]
y_val = y_all[val_idx]
X_test_seq_full = X_scaled[test_idx]
y_test = y_all[test_idx]

print(f"Train: {X_train_raw_seq.shape}")  # should be (29660, 1034) — full sequence, not 42
print(f"Val:   {X_val_seq_full.shape}")
print(f"Test:  {X_test_seq_full.shape}")

# SMOTE on the full-sequence training data
smote = SMOTE(random_state=SEED)
X_train_smote_seq, y_train_smote_seq = smote.fit_resample(X_train_raw_seq, y_train_raw)

# Reshape for GRU: (samples, timesteps, 1)
X_tr_seq = X_train_smote_seq.reshape(X_train_smote_seq.shape[0], X_train_smote_seq.shape[1], 1)
X_val_seq = X_val_seq_full.reshape(X_val_seq_full.shape[0], X_val_seq_full.shape[1], 1)
X_test_seq = X_test_seq_full.reshape(X_test_seq_full.shape[0], X_test_seq_full.shape[1], 1)

print(f"GRU input shape: {X_tr_seq.shape}  (samples, {X_tr_seq.shape[1]} timesteps, 1 feature)")

Train: (29660, 1034)
Val:   (6356, 1034)
Test:  (6356, 1034)
GRU input shape: (54260, 1034, 1)  (samples, 1034 timesteps, 1 feature)


In [ ]:
print(X_train_raw_seq.shape)  # should be (29660, 1034) — pre-SMOTE
print(poisoned_labels_raw[0.0].shape)  # should be (29660,)

(29660, 1034)
(29660,)


In [ ]:
from tensorflow.keras.constraints import MaxNorm
from tensorflow.keras.utils import to_categorical

def build_gru_exact_takiddin():
    model = keras.Sequential()
    model.add(layers.Input(shape=(X_tr_seq.shape[1], 1)))

    for i in range(8):  # L = 8, from Table II
        return_seq = (i < 7)
        model.add(layers.GRU(
            300,                            # N = 300
            return_sequences=return_seq,
            kernel_constraint=MaxNorm(5),   # G = 5
            recurrent_constraint=MaxNorm(5),
            activation='relu'               # A_H = ReLU
        ))
        model.add(layers.Dropout(0.2))      # B = 0.2

    model.add(layers.Dense(2, activation='softmax'))  # A_O = Softmax, 2-neuron

    model.compile(
        optimizer=keras.optimizers.Adam(),
        loss='categorical_crossentropy',
        metrics=[keras.metrics.AUC(name='auc')]
    )
    return model

# One-hot encode for 2-neuron softmax
y_train_smote_oh = to_categorical(y_train_smote_seq, num_classes=2)
y_val_oh = to_categorical(y_val, num_classes=2)
y_test_oh = to_categorical(y_test, num_classes=2)

model = build_gru_exact_takiddin()
model.summary()

print("\nStarting training — this will take a while given 8 layers x 300 cells x 1034 timesteps")
t0 = time.time()
history = model.fit(
    X_tr_seq, y_train_smote_oh,
    validation_data=(X_val_seq, y_val_oh),
    epochs=30,
    batch_size=100,   # K=100, matches Takiddin exactly
    callbacks=[EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1)],
    verbose=1
)
elapsed = time.time() - t0

proba = model.predict(X_test_seq, verbose=0)[:, 1]
pred = np.argmax(model.predict(X_test_seq, verbose=0), axis=1)

print(f"\n{'='*50}")
print(f"Exact-match GRU — TEST SET (0% poison)")
print(f"{'='*50}")
print(f"  F1:  {f1_score(y_test, pred):.4f}")
print(f"  DR:  {recall_score(y_test, pred):.4f}")
print(f"  AUC: {roc_auc_score(y_test, proba):.4f}")
print(f"  Time: {elapsed:.1f}s ({elapsed/60:.1f} min)")
print(f"\nTakiddin paper target (Irish dataset): DR=0.924, F1=0.923, AUC=0.921")

# Save immediately
import joblib
model.save(f'{SAVE_DIR}/gru_exact_takiddin_0pct.keras')
with open(f'{SAVE_DIR}/gru_exact_result.json', 'w') as f:
    json.dump({'F1': float(f1_score(y_test, pred)), 'DR': float(recall_score(y_test, pred)),
               'AUC': float(roc_auc_score(y_test, proba)), 'time_s': elapsed}, f, indent=2)
print("Saved model and results to Drive")

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ gru (GRU)                       │ (None, 1034, 300)      │       272,700 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1034, 300)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_1 (GRU)                     │ (None, 1034, 300)      │       541,800 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 1034, 300)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_2 (GRU)                     │ (None, 1034, 300)      │       541,800 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 1034, 300)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_3 (GRU)                     │ (None, 1034, 300)      │       541,800 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 1034, 300)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_4 (GRU)                     │ (None, 1034, 300)      │       541,800 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 1034, 300)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_5 (GRU)                     │ (None, 1034, 300)      │       541,800 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 1034, 300)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_6 (GRU)                     │ (None, 1034, 300)      │       541,800 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_6 (Dropout)             │ (None, 1034, 300)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_7 (GRU)                     │ (None, 300)            │       541,800 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_7 (Dropout)             │ (None, 300)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 2)              │           602 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,065,902 (15.51 MB)

 Trainable params: 4,065,902 (15.51 MB)

 Non-trainable params: 0 (0.00 B)


Starting training — this will take a while given 8 layers x 300 cells x 1034 timesteps
Epoch 1/30


KeyboardInterrupt: 

In [ ]:
# Interrupt training, then run this to see what the checkpoint would have restored
print(f"Best val_auc observed: 0.6411 (epoch 4)")
print(f"This is what restore_best_weights=True would have kept")

Best val_auc observed: 0.6411 (epoch 4)
This is what restore_best_weights=True would have kept


In [ ]:
def build_gru_tuned():
    model = keras.Sequential()
    model.add(layers.Input(shape=(X_tr_seq.shape[1], 1)))

    for i in range(6):  # from random search winner
        return_seq = (i < 5)
        model.add(layers.GRU(300, return_sequences=return_seq))
        model.add(layers.Dropout(0.2))

    model.add(layers.Dense(1, activation='sigmoid'))  # binary, not softmax — simpler, stable

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.001),
        loss='binary_crossentropy',
        metrics=[keras.metrics.AUC(name='auc')]
    )
    return model

gru_poison_results = []

print("GRU (tuned config) — All Poison Levels")
print("=" * 50)

for rate in POISON_RATES:
    y_poisoned = poisoned_labels_raw[rate]
    smote = SMOTE(random_state=SEED)
    X_tr_s, y_tr_s = smote.fit_resample(X_train_raw_seq, y_poisoned)
    X_tr_s_seq = X_tr_s.reshape(X_tr_s.shape[0], X_tr_s.shape[1], 1)

    model = build_gru_tuned()
    t0 = time.time()
    model.fit(
        X_tr_s_seq, y_tr_s,
        validation_data=(X_val_seq, y_val),
        epochs=30,
        batch_size=256,
        callbacks=[EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=0)],
        verbose=1
    )
    elapsed = time.time() - t0

    proba = model.predict(X_test_seq, verbose=0).flatten()
    pred = (proba >= 0.5).astype(int)

    result = evaluate(y_test, pred, proba, 'GRU_tuned', rate, elapsed)
    gru_poison_results.append(result)

    print(f'  {int(rate*100):>2}% — F1={result["F1"]:.4f}  DR={result["DR"]:.4f}  '
          f'AUC={result["AUC"]:.4f}  MCC={result["MCC"]:.4f}  ({elapsed/60:.1f} min)')

gru_df = pd.DataFrame([{k:v for k,v in r.items() if k in cols} for r in gru_poison_results])
gru_df.to_csv(f'{SAVE_DIR}/gru_tuned_poison_results.csv', index=False)
print('\nSaved: gru_tuned_poison_results.csv')

GRU (tuned config) — All Poison Levels
Epoch 1/30
212/212 ━━━━━━━━━━━━━━━━━━━━ 73s 324ms/step - auc: 0.6641 - loss: 0.6486 - val_auc: 0.6543 - val_loss: 0.7409
Epoch 2/30
212/212 ━━━━━━━━━━━━━━━━━━━━ 68s 320ms/step - auc: 0.6688 - loss: 0.6461 - val_auc: 0.6543 - val_loss: 0.7261
Epoch 3/30
212/212 ━━━━━━━━━━━━━━━━━━━━ 68s 320ms/step - auc: 0.6707 - loss: 0.6447 - val_auc: 0.6534 - val_loss: 0.7230
Epoch 4/30
212/212 ━━━━━━━━━━━━━━━━━━━━ 68s 320ms/step - auc: 0.6717 - loss: 0.6438 - val_auc: 0.6533 - val_loss: 0.7286
Epoch 5/30
212/212 ━━━━━━━━━━━━━━━━━━━━ 68s 321ms/step - auc: 0.6746 - loss: 0.6419 - val_auc: 0.6523 - val_loss: 0.7054
Epoch 6/30
212/212 ━━━━━━━━━━━━━━━━━━━━ 68s 321ms/step - auc: 0.6804 - loss: 0.6390 - val_auc: 0.6474 - val_loss: 0.7530
Epoch 7/30
212/212 ━━━━━━━━━━━━━━━━━━━━ 68s 320ms/step - auc: 0.6939 - loss: 0.6299 - val_auc: 0.6510 - val_loss: 0.7372
Epoch 8/30
212/212 ━━━━━━━━━━━━━━━━━━━━ 68s 320ms/step - auc: 0.7297 - loss: 0.6067 - val_auc: 0.6871 - val_loss: 

In [ ]:
def build_gru_from_search(input_shape):
    model = keras.Sequential()
    model.add(layers.Input(shape=input_shape))
    for i in range(6):
        return_seq = (i < 5)
        model.add(layers.GRU(300, return_sequences=return_seq))
        if 0.2 > 0:
            model.add(layers.Dropout(0.2))
    model.add(layers.Dense(1, activation='sigmoid'))
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.001),
        loss='binary_crossentropy',
        metrics=[keras.metrics.AUC(name='auc')]
    )
    return model

# X_train_raw here = the PCA-42 version (confirm shape is (29660, 42))
print("Checking X_train_raw shape (should be PCA-42, not raw sequence):", X_train_raw.shape)

smote = SMOTE(random_state=SEED)
X_tr_pca_smote, y_tr_pca_smote = smote.fit_resample(X_train_raw, y_train_raw)
X_tr_pca_seq = X_tr_pca_smote.reshape(X_tr_pca_smote.shape[0], X_tr_pca_smote.shape[1], 1)
X_val_pca_seq = X_val.reshape(X_val.shape[0], X_val.shape[1], 1)
X_test_pca_seq = X_test.reshape(X_test.shape[0], X_test.shape[1], 1)

final_gru = build_gru_from_search((X_tr_pca_seq.shape[1], 1))
final_gru.fit(X_tr_pca_seq, y_tr_pca_smote, validation_data=(X_val_pca_seq, y_val),
              epochs=30, batch_size=256,
              callbacks=[EarlyStopping(patience=5, restore_best_weights=True, verbose=0)],
              verbose=1)

proba = final_gru.predict(X_test_pca_seq, verbose=0).flatten()
pred = (proba >= 0.5).astype(int)
print(f"\nTuned GRU on PCA-42 (correct input) — TEST SET:")
print(f"  F1: {f1_score(y_test, pred):.4f}  AUC: {roc_auc_score(y_test, pred if False else proba):.4f}")

Checking X_train_raw shape (should be PCA-42, not raw sequence): (29660, 42)
Epoch 1/30
212/212 ━━━━━━━━━━━━━━━━━━━━ 10s 28ms/step - auc: 0.5609 - loss: 0.6815 - val_auc: 0.6285 - val_loss: 0.6778
Epoch 2/30
212/212 ━━━━━━━━━━━━━━━━━━━━ 5s 21ms/step - auc: 0.6131 - loss: 0.6600 - val_auc: 0.6681 - val_loss: 0.6610
Epoch 3/30
212/212 ━━━━━━━━━━━━━━━━━━━━ 5s 21ms/step - auc: 0.6477 - loss: 0.6530 - val_auc: 0.6886 - val_loss: 0.6830
Epoch 4/30
212/212 ━━━━━━━━━━━━━━━━━━━━ 5s 21ms/step - auc: 0.6986 - loss: 0.6296 - val_auc: 0.7147 - val_loss: 0.6319
Epoch 5/30
212/212 ━━━━━━━━━━━━━━━━━━━━ 5s 21ms/step - auc: 0.7207 - loss: 0.6138 - val_auc: 0.7230 - val_loss: 0.6697
Epoch 6/30
212/212 ━━━━━━━━━━━━━━━━━━━━ 5s 21ms/step - auc: 0.7361 - loss: 0.6024 - val_auc: 0.7336 - val_loss: 0.6641
Epoch 7/30
212/212 ━━━━━━━━━━━━━━━━━━━━ 5s 23ms/step - auc: 0.7486 - loss: 0.5926 - val_auc: 0.7280 - val_loss: 0.6413
Epoch 8/30
212/212 ━━━━━━━━━━━━━━━━━━━━ 5s 22ms/step - auc: 0.7667 - loss: 0.5772 - val_a

In [ ]:
gru_final_results = []
print("GRU (tuned, PCA-42) — All Poison Levels")
print("=" * 50)

for rate in POISON_RATES:
    y_poisoned = poisoned_labels_raw[rate] if 'poisoned_labels_raw' in dir() else inject_poison(y_train_raw, rate)
    smote = SMOTE(random_state=SEED)
    X_tr_s, y_tr_s = smote.fit_resample(X_train_raw, y_poisoned)
    X_tr_s_seq = X_tr_s.reshape(X_tr_s.shape[0], X_tr_s.shape[1], 1)

    model = build_gru_from_search((X_tr_s_seq.shape[1], 1))
    t0 = time.time()
    model.fit(X_tr_s_seq, y_tr_s, validation_data=(X_val_pca_seq, y_val),
              epochs=30, batch_size=256,
              callbacks=[EarlyStopping(patience=5, restore_best_weights=True, verbose=0)],
              verbose=0)
    elapsed = time.time() - t0

    proba = model.predict(X_test_pca_seq, verbose=0).flatten()
    pred = (proba >= 0.5).astype(int)
    result = evaluate(y_test, pred, proba, 'GRU_final', rate, elapsed)
    gru_final_results.append(result)
    print(f'  {int(rate*100):>2}% — F1={result["F1"]:.4f}  DR={result["DR"]:.4f}  AUC={result["AUC"]:.4f}  ({elapsed:.0f}s)')

gru_final_df = pd.DataFrame([{k:v for k,v in r.items() if k in cols} for r in gru_final_results])
gru_final_df.to_csv(f'{SAVE_DIR}/gru_final_poison_results.csv', index=False)
print('\nSaved: gru_final_poison_results.csv')

GRU (tuned, PCA-42) — All Poison Levels
   0% — F1=0.2614  DR=0.5387  AUC=0.6851  (81s)
  10% — F1=0.0593  DR=0.0314  AUC=0.5375  (46s)
  20% — F1=0.1575  DR=0.1402  AUC=0.5254  (31s)
  30% — F1=0.1548  DR=0.9797  AUC=0.5608  (33s)

Saved: gru_final_poison_results.csv


In [ ]:
# Only run this if the shape check above showed 1034 instead of 42
X_train_raw = X_pca[train_idx]  # requires X_pca still in memory from earlier
print(X_train_raw.shape)

(29660, 42)


In [ ]:
# Add this near the top of your GRU build function, before defining layers
import os
os.environ['TF_USE_CUDNN_RNN'] = '0'

## 9. Deep Learning — AEA Hyperparameter Tuning with exact-match Takiddin architecture


In [ ]:
from sklearn.utils.class_weight import compute_class_weight

# Confirm we're using PCA-42 input (proven to work better than raw sequence)
print("X_train_raw shape:", X_train_raw.shape)  # should be (29660, 42)

class_weights = compute_class_weight('balanced', classes=np.array([0,1]), y=y_train_raw)
class_weight_dict = {0: class_weights[0], 1: class_weights[1]}
print(f"Class weights: {class_weight_dict}")

X_val_pca_seq = X_val.reshape(X_val.shape[0], X_val.shape[1], 1)
X_test_pca_seq = X_test.reshape(X_test.shape[0], X_test.shape[1], 1)

X_train_raw shape: (29660, 42)
Class weights: {0: np.float64(0.5466273497972723), 1: np.float64(5.861660079051384)}


In [ ]:
from tensorflow.keras.constraints import MaxNorm

def build_aea_exact(input_shape):
    inputs = keras.Input(shape=input_shape)

    x = layers.LSTM(500, return_sequences=True, kernel_constraint=MaxNorm(1))(inputs)
    x = layers.LSTM(300, return_sequences=True, kernel_constraint=MaxNorm(1))(x)
    x = layers.LSTM(200, return_sequences=True, kernel_constraint=MaxNorm(1))(x)

    attention = layers.Dense(1, activation='tanh')(x)
    attention = layers.Flatten()(attention)
    attention_weights = layers.Activation('softmax')(attention)
    attention_weights = layers.RepeatVector(200)(attention_weights)
    attention_weights = layers.Permute([2, 1])(attention_weights)
    x = layers.Multiply()([x, attention_weights])

    x = layers.LSTM(200, return_sequences=True, kernel_constraint=MaxNorm(1))(x)
    x = layers.LSTM(300, return_sequences=True, kernel_constraint=MaxNorm(1))(x)
    x = layers.LSTM(500, return_sequences=False, kernel_constraint=MaxNorm(1))(x)

    outputs = layers.Dense(1, activation='sigmoid')(x)

    model = Model(inputs, outputs)
    model.compile(
        optimizer=keras.optimizers.SGD(learning_rate=0.01, momentum=0.9),
        loss='binary_crossentropy',
        metrics=[keras.metrics.AUC(name='auc')]
    )
    return model

test_model = build_aea_exact((X_train_raw.shape[1], 1))
test_model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 42, 1)     │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm (LSTM)         │ (None, 42, 500)   │  1,004,000 │ input_layer_1[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_1 (LSTM)       │ (None, 42, 300)   │    961,200 │ lstm[0][0]        │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_2 (LSTM)       │ (None, 42, 200)   │    400,800 │ lstm_1[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 42, 1)     │        201 │ lstm_2[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_1 (Flatten) │ (None, 42)        │          0 │ dense_2[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_1        │ (None, 42)        │          0 │ flatten_1[0][0]   │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ repeat_vector_1     │ (None, 200, 42)   │          0 │ activation_1[0][… │
│ (RepeatVector)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ permute_1 (Permute) │ (None, 42, 200)   │          0 │ repeat_vector_1[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multiply_1          │ (None, 42, 200)   │          0 │ lstm_2[0][0],     │
│ (Multiply)          │                   │            │ permute_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_3 (LSTM)       │ (None, 42, 200)   │    320,800 │ multiply_1[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_4 (LSTM)       │ (None, 42, 300)   │    601,200 │ lstm_3[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_5 (LSTM)       │ (None, 500)       │  1,602,000 │ lstm_4[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 1)         │        501 │ lstm_5[0][0]      │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 4,890,702 (18.66 MB)

 Trainable params: 4,890,702 (18.66 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
X_val_pca_seq = X_val.reshape(X_val.shape[0], X_val.shape[1], 1)
X_test_pca_seq = X_test.reshape(X_test.shape[0], X_test.shape[1], 1)

X_tr_seq_raw = X_train_raw.reshape(X_train_raw.shape[0], X_train_raw.shape[1], 1)

model = build_aea_exact((X_tr_seq_raw.shape[1], 1))

t0 = time.time()
history = model.fit(
    X_tr_seq_raw, y_train_raw,   # no SMOTE — class_weight handles imbalance
    class_weight=class_weight_dict,
    validation_data=(X_val_pca_seq, y_val),
    epochs=30,
    batch_size=256,
    callbacks=[EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1)],
    verbose=1
)
elapsed = time.time() - t0

proba = model.predict(X_test_pca_seq, verbose=0).flatten()
pred = (proba >= 0.5).astype(int)

print(f"\n{'='*50}")
print(f"AEA (exact-match) — TEST SET (0% poison)")
print(f"{'='*50}")
print(f"  F1:  {f1_score(y_test, pred):.4f}")
print(f"  DR:  {recall_score(y_test, pred):.4f}")
print(f"  AUC: {roc_auc_score(y_test, proba):.4f}")
print(f"  Time: {elapsed:.1f}s")
print(f"\nTakiddin paper target (Irish dataset, generalized AEA): DR=0.941, F1=0.943, AUC=0.940")

Epoch 1/30
116/116 ━━━━━━━━━━━━━━━━━━━━ 23s 148ms/step - auc: 0.5022 - loss: 0.6933 - val_auc: 0.5000 - val_loss: 0.7154
Epoch 2/30
116/116 ━━━━━━━━━━━━━━━━━━━━ 17s 148ms/step - auc: 0.5038 - loss: 0.6934 - val_auc: 0.5000 - val_loss: 0.7146
Epoch 3/30
116/116 ━━━━━━━━━━━━━━━━━━━━ 19s 161ms/step - auc: 0.5033 - loss: 0.6934 - val_auc: 0.5000 - val_loss: 0.7141
Epoch 4/30
116/116 ━━━━━━━━━━━━━━━━━━━━ 21s 179ms/step - auc: 0.5028 - loss: 0.6934 - val_auc: 0.5000 - val_loss: 0.7135
Epoch 5/30
116/116 ━━━━━━━━━━━━━━━━━━━━ 20s 171ms/step - auc: 0.5022 - loss: 0.6934 - val_auc: 0.5000 - val_loss: 0.7130
Epoch 6/30
116/116 ━━━━━━━━━━━━━━━━━━━━ 19s 161ms/step - auc: 0.5024 - loss: 0.6934 - val_auc: 0.5000 - val_loss: 0.7125
Epoch 7/30
116/116 ━━━━━━━━━━━━━━━━━━━━ 19s 166ms/step - auc: 0.5020 - loss: 0.6934 - val_auc: 0.5000 - val_loss: 0.7121
Epoch 8/30
 82/116 ━━━━━━━━━━━━━━━━━━━━ 5s 161ms/step - auc: 0.4972 - loss: 0.7005

KeyboardInterrupt: 

In [ ]:
def build_aea_reconstruction(input_shape):
    inputs = keras.Input(shape=input_shape)

    # Encoder: 500 -> 300 -> 200
    x = layers.LSTM(500, return_sequences=True, kernel_constraint=MaxNorm(1))(inputs)
    x = layers.LSTM(300, return_sequences=True, kernel_constraint=MaxNorm(1))(x)
    encoded = layers.LSTM(200, return_sequences=True, kernel_constraint=MaxNorm(1))(x)

    # Attention over encoder output
    attention = layers.Dense(1, activation='tanh')(encoded)
    attention = layers.Flatten()(attention)
    attention_weights = layers.Activation('softmax')(attention)
    attention_weights = layers.RepeatVector(200)(attention_weights)
    attention_weights = layers.Permute([2, 1])(attention_weights)
    context = layers.Multiply()([encoded, attention_weights])

    # Decoder: 200 -> 300 -> 500, reconstructing the ORIGINAL input shape
    x = layers.LSTM(200, return_sequences=True, kernel_constraint=MaxNorm(1))(context)
    x = layers.LSTM(300, return_sequences=True, kernel_constraint=MaxNorm(1))(x)
    x = layers.LSTM(500, return_sequences=True, kernel_constraint=MaxNorm(1))(x)

    # Output layer reconstructs the input — same feature dimension, per timestep
    reconstructed = layers.TimeDistributed(layers.Dense(input_shape[-1]))(x)

    model = Model(inputs, reconstructed)
    model.compile(
        optimizer=keras.optimizers.SGD(learning_rate=0.01, momentum=0.9),
        loss='mse'   # reconstruction error
    )
    return model

# Train ONLY on benign (y=0) samples — this is the critical difference
X_train_benign = X_train_raw[y_train_raw == 0]
X_train_benign_seq = X_train_benign.reshape(X_train_benign.shape[0], X_train_benign.shape[1], 1)

print(f"Training on benign-only data: {X_train_benign_seq.shape}")
print(f"(Excluded {(y_train_raw==1).sum()} theft samples from training, as paper specifies)")

model = build_aea_reconstruction((X_train_benign_seq.shape[1], 1))
model.summary()

Training on benign-only data: (27130, 42, 1)
(Excluded 2530 theft samples from training, as paper specifies)


Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_3       │ (None, 42, 1)     │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_12 (LSTM)      │ (None, 42, 500)   │  1,004,000 │ input_layer_3[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_13 (LSTM)      │ (None, 42, 300)   │    961,200 │ lstm_12[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_14 (LSTM)      │ (None, 42, 200)   │    400,800 │ lstm_13[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_6 (Dense)     │ (None, 42, 1)     │        201 │ lstm_14[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_3 (Flatten) │ (None, 42)        │          0 │ dense_6[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_3        │ (None, 42)        │          0 │ flatten_3[0][0]   │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ repeat_vector_3     │ (None, 200, 42)   │          0 │ activation_3[0][… │
│ (RepeatVector)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ permute_3 (Permute) │ (None, 42, 200)   │          0 │ repeat_vector_3[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multiply_3          │ (None, 42, 200)   │          0 │ lstm_14[0][0],    │
│ (Multiply)          │                   │            │ permute_3[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_15 (LSTM)      │ (None, 42, 200)   │    320,800 │ multiply_3[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_16 (LSTM)      │ (None, 42, 300)   │    601,200 │ lstm_15[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_17 (LSTM)      │ (None, 42, 500)   │  1,602,000 │ lstm_16[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ time_distributed    │ (None, 42, 1)     │        501 │ lstm_17[0][0]     │
│ (TimeDistributed)   │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 4,890,702 (18.66 MB)

 Trainable params: 4,890,702 (18.66 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
t0 = time.time()
history = model.fit(
    X_train_benign_seq, X_train_benign_seq,   # reconstruct itself — autoencoder target = input
    validation_split=0.1,   # small internal validation for early stopping only
    epochs=30,
    batch_size=256,
    callbacks=[EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1)],
    verbose=1
)
elapsed = time.time() - t0
print(f"\nTraining time: {elapsed:.1f}s")

Epoch 1/30
96/96 ━━━━━━━━━━━━━━━━━━━━ 23s 182ms/step - loss: 3.5899 - val_loss: 6.5298
Epoch 2/30
96/96 ━━━━━━━━━━━━━━━━━━━━ 18s 186ms/step - loss: 3.5863 - val_loss: 6.5293
Epoch 3/30
96/96 ━━━━━━━━━━━━━━━━━━━━ 17s 179ms/step - loss: 3.5849 - val_loss: 6.5290
Epoch 4/30
96/96 ━━━━━━━━━━━━━━━━━━━━ 16s 164ms/step - loss: 3.5838 - val_loss: 6.5287
Epoch 5/30
96/96 ━━━━━━━━━━━━━━━━━━━━ 15s 160ms/step - loss: 3.5830 - val_loss: 6.5285
Epoch 6/30
65/96 ━━━━━━━━━━━━━━━━━━━━ 4s 159ms/step - loss: 3.8339

KeyboardInterrupt: 

In [ ]:
def build_aea_reconstruction_v2(input_shape):
    inputs = keras.Input(shape=input_shape)

    x = layers.LSTM(500, return_sequences=True, kernel_constraint=MaxNorm(1))(inputs)
    x = layers.LSTM(300, return_sequences=True, kernel_constraint=MaxNorm(1))(x)
    encoded = layers.LSTM(200, return_sequences=True, kernel_constraint=MaxNorm(1))(x)

    attention = layers.Dense(1, activation='tanh')(encoded)
    attention = layers.Flatten()(attention)
    attention_weights = layers.Activation('softmax')(attention)
    attention_weights = layers.RepeatVector(200)(attention_weights)
    attention_weights = layers.Permute([2, 1])(attention_weights)
    context = layers.Multiply()([encoded, attention_weights])

    x = layers.LSTM(200, return_sequences=True, kernel_constraint=MaxNorm(1))(context)
    x = layers.LSTM(300, return_sequences=True, kernel_constraint=MaxNorm(1))(x)
    x = layers.LSTM(500, return_sequences=True, kernel_constraint=MaxNorm(1))(x)

    reconstructed = layers.TimeDistributed(layers.Dense(input_shape[-1]))(x)

    model = Model(inputs, reconstructed)
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.001),  # DEVIATION: paper specifies SGD, which failed to converge
        loss='mse'
    )
    return model

model = build_aea_reconstruction_v2((X_train_benign_seq.shape[1], 1))

t0 = time.time()
history = model.fit(
    X_train_benign_seq, X_train_benign_seq,
    validation_split=0.1,
    epochs=30,
    batch_size=256,
    callbacks=[EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1)],
    verbose=1
)
elapsed = time.time() - t0
print(f"\nTraining time: {elapsed:.1f}s")

Epoch 1/30
96/96 ━━━━━━━━━━━━━━━━━━━━ 25s 191ms/step - loss: 3.2044 - val_loss: 4.2955
Epoch 2/30
96/96 ━━━━━━━━━━━━━━━━━━━━ 18s 187ms/step - loss: 1.3285 - val_loss: 3.3308
Epoch 3/30
96/96 ━━━━━━━━━━━━━━━━━━━━ 16s 169ms/step - loss: 0.9586 - val_loss: 3.0390
Epoch 4/30
96/96 ━━━━━━━━━━━━━━━━━━━━ 16s 163ms/step - loss: 0.7615 - val_loss: 2.7985
Epoch 5/30
96/96 ━━━━━━━━━━━━━━━━━━━━ 16s 167ms/step - loss: 0.8405 - val_loss: 2.8687
Epoch 6/30
96/96 ━━━━━━━━━━━━━━━━━━━━ 17s 180ms/step - loss: 0.6496 - val_loss: 2.7244
Epoch 7/30
96/96 ━━━━━━━━━━━━━━━━━━━━ 17s 176ms/step - loss: 0.6144 - val_loss: 2.4671
Epoch 8/30
96/96 ━━━━━━━━━━━━━━━━━━━━ 16s 168ms/step - loss: 0.4393 - val_loss: 2.3344
Epoch 9/30
96/96 ━━━━━━━━━━━━━━━━━━━━ 16s 168ms/step - loss: 0.4281 - val_loss: 2.2908
Epoch 10/30
96/96 ━━━━━━━━━━━━━━━━━━━━ 16s 171ms/step - loss: 0.4028 - val_loss: 2.5871
Epoch 11/30
96/96 ━━━━━━━━━━━━━━━━━━━━ 17s 175ms/step - loss: 0.4341 - val_loss: 2.2990
Epoch 12/30
96/96 ━━━━━━━━━━━━━━━━━━━━ 17

In [ ]:
X_val_seq_full = X_val.reshape(X_val.shape[0], X_val.shape[1], 1)
val_reconstructed = model.predict(X_val_seq_full, verbose=0)
val_recon_error = np.mean(np.square(X_val_seq_full - val_reconstructed), axis=(1,2))

fpr_val, tpr_val, roc_thresholds = roc_curve(y_val, val_recon_error)

q1_idx = int(len(roc_thresholds) * 0.25)
q3_idx = int(len(roc_thresholds) * 0.75)
iqr_thresholds = roc_thresholds[q1_idx:q3_idx]
computed_threshold = np.median(iqr_thresholds)

print(f"Computed threshold: {computed_threshold:.6f}")
print(f"Reconstruction error stats — Val set:")
print(f"  Benign mean error: {val_recon_error[y_val==0].mean():.4f}")
print(f"  Theft mean error:  {val_recon_error[y_val==1].mean():.4f}")

X_test_seq_full = X_test.reshape(X_test.shape[0], X_test.shape[1], 1)
test_reconstructed = model.predict(X_test_seq_full, verbose=0)
test_recon_error = np.mean(np.square(X_test_seq_full - test_reconstructed), axis=(1,2))

pred = (test_recon_error >= computed_threshold).astype(int)

print(f"\nAEA (reconstruction-based, Adam) — TEST SET (0% poison)")
print(f"  F1:  {f1_score(y_test, pred):.4f}")
print(f"  DR:  {recall_score(y_test, pred):.4f}")
print(f"  AUC: {roc_auc_score(y_test, test_recon_error):.4f}")
print(f"\nTakiddin paper target: DR=0.941, F1=0.943, AUC=0.940")

Computed threshold: 0.005556
Reconstruction error stats — Val set:
  Benign mean error: 0.2769
  Theft mean error:  97.2044

AEA (reconstruction-based, Adam) — TEST SET (0% poison)
  F1:  0.1746
  DR:  0.5092
  AUC: 0.5915

Takiddin paper target: DR=0.941, F1=0.943, AUC=0.940


## 9. Deep Learning — Wide & Deep CNN Hyperparameter Tuning

Architecture from Zheng et al. (2018) — the paper that released SGCC.
Requires raw sequences reshaped into weekly 2D matrix + 1D wide input.

In [ ]:
import keras_tuner as kt

In [ ]:
# Prepare raw (non-PCA) data for Wide & Deep CNN
# This model needs the original daily readings, not PCA
X_raw_imputed = imputer.transform(features_raw)
X_raw_scaled = scaler.transform(X_raw_imputed)

# Split using same indices
X_train_raw_full = X_raw_scaled[train_idx]
X_val_raw_full = X_raw_scaled[val_idx]
X_test_raw_full = X_raw_scaled[test_idx]

# Reshape to weekly 2D: 1034 days -> 1029 days (147 weeks x 7 days)
N_DAYS = 1029
N_WEEKS = 147

def prepare_wide_deep(X_full):
    x_1d = X_full[:, :N_DAYS]
    x_2d = X_full[:, :N_DAYS].reshape(X_full.shape[0], N_WEEKS, 7)
    return x_1d, x_2d

# SMOTE on 1D version, then reshape both
smote_wd = SMOTE(random_state=SEED)
X_tr_1d_raw = X_train_raw_full[:, :N_DAYS]
X_tr_1d_smote, y_tr_wd_smote = smote_wd.fit_resample(X_tr_1d_raw, y_train_raw)
X_tr_2d_smote = X_tr_1d_smote.reshape(X_tr_1d_smote.shape[0], N_WEEKS, 7)

X_val_1d, X_val_2d = prepare_wide_deep(X_val_raw_full)
X_test_1d, X_test_2d = prepare_wide_deep(X_test_raw_full)

print(f"Wide&Deep train 1D: {X_tr_1d_smote.shape}, 2D: {X_tr_2d_smote.shape}")
print(f"Wide&Deep test 1D:  {X_test_1d.shape}, 2D: {X_test_2d.shape}")

def build_wide_deep(hp):
    alpha = hp.Choice('alpha', [60, 90, 120])
    beta = hp.Choice('beta', [30, 60, 90])
    gamma = hp.Choice('gamma', [10, 15, 20])
    R = hp.Choice('R', [3, 5, 7])
    lr = hp.Choice('learning_rate', [0.001, 0.0005, 0.0001])

    input_wide = keras.Input(shape=(N_DAYS,), name='input_1d')
    wide_out = layers.Dense(alpha, activation='relu')(input_wide)

    input_deep = keras.Input(shape=(N_WEEKS, 7), name='input_2d')
    x = layers.Reshape((N_WEEKS, 7, 1))(input_deep)
    for r in range(R):
        x = layers.Conv2D(gamma, (3, 3), padding='same', activation='tanh')(x)
    x = layers.MaxPooling2D((2, 2))(x)
    x = layers.Flatten()(x)
    deep_out = layers.Dense(beta, activation='relu')(x)

    combined = layers.Concatenate()([wide_out, deep_out])
    output = layers.Dense(1, activation='sigmoid')(combined)

    model = Model(inputs=[input_wide, input_deep], outputs=output)
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr),
        loss='binary_crossentropy',
        metrics=[keras.metrics.AUC(name='auc')]
    )
    return model

wd_tuner = kt.RandomSearch(
    build_wide_deep,
    objective=kt.Objective('val_auc', direction='max'),
    max_trials=20,
    executions_per_trial=1,
    directory=f'{SAVE_DIR}/wd_tuning',
    project_name='wide_deep',
    seed=SEED
)

wd_tuner.search(
    [X_tr_1d_smote, X_tr_2d_smote], y_tr_wd_smote,
    validation_data=([X_val_1d, X_val_2d], y_val),
    epochs=30,
    batch_size=256,
    callbacks=[EarlyStopping(patience=5, restore_best_weights=True, verbose=0)],
    verbose=1
)

best_wd_hp = wd_tuner.get_best_hyperparameters(1)[0]
print(f"\nBest Wide&Deep config: {best_wd_hp.values}")

with open(f'{SAVE_DIR}/wd_best_hp.json', 'w') as f:
    json.dump(best_wd_hp.values, f, indent=2)
print("Wide&Deep tuning complete — saved to Drive")

In [ ]:
X_1d_train = X_train_raw_full[:, :N_DAYS] if 'X_train_raw_full' in dir() else None
# If X_train_raw_full doesn't exist (lost to runtime switch), rebuild:
if X_1d_train is None:
    X_raw_imputed = imputer.transform(features_raw)
    X_raw_scaled = scaler.transform(X_raw_imputed)
    X_train_raw_full = X_raw_scaled[train_idx]
    X_val_raw_full = X_raw_scaled[val_idx]
    X_test_raw_full = X_raw_scaled[test_idx]

N_DAYS = 1029
N_WEEKS = 147

X_tr_1d = X_train_raw_full[:, :N_DAYS]
X_tr_2d = X_tr_1d.reshape(-1, N_WEEKS, 7)
X_val_1d, X_val_2d = X_val_raw_full[:, :N_DAYS], X_val_raw_full[:, :N_DAYS].reshape(-1, N_WEEKS, 7)

wd_tuner = kt.RandomSearch(
    build_wide_deep,
    objective=kt.Objective('val_auc', direction='max'),
    max_trials=10,   # reduced from 20, given time constraints and established pattern
    directory=f'{SAVE_DIR}/wd_tuning',
    project_name='wide_deep',
    seed=SEED
)

wd_tuner.search(
    [X_tr_1d, X_tr_2d], y_train_raw,
    validation_data=([X_val_1d, X_val_2d], y_val),
    epochs=20,
    batch_size=256,
    callbacks=[EarlyStopping(patience=5, restore_best_weights=True)],
    verbose=1
)

best_wd_hp = wd_tuner.get_best_hyperparameters(1)[0]
print(f"Best Wide&Deep config: {best_wd_hp.values}")

In [ ]:
required_vars = ['X_train_raw_full', 'X_val_raw_full', 'X_test_raw_full', 'N_DAYS', 'N_WEEKS',
                  'X_tr_1d', 'X_tr_2d', 'X_val_1d', 'X_val_2d', 'y_train_raw', 'y_val',
                  'build_wide_deep', 'kt', 'EarlyStopping', 'Model', 'layers', 'keras']
for v in required_vars:
    print(f"{v}: {'OK' if v in dir() else 'MISSING'}")

X_train_raw_full: OK
X_val_raw_full: OK
X_test_raw_full: OK
N_DAYS: OK
N_WEEKS: OK
X_tr_1d: OK
X_tr_2d: OK
X_val_1d: OK
X_val_2d: OK
y_train_raw: OK
y_val: OK
build_wide_deep: MISSING
kt: OK
EarlyStopping: OK
Model: OK
layers: OK
keras: OK


In [ ]:
def build_wide_deep(hp):
    alpha = hp.Choice('alpha', [60, 90, 120])
    beta = hp.Choice('beta', [30, 60, 90])
    gamma = hp.Choice('gamma', [10, 15, 20])
    R = hp.Choice('R', [3, 5, 7])
    lr = hp.Choice('learning_rate', [0.001, 0.0005, 0.0001])

    input_wide = keras.Input(shape=(N_DAYS,), name='input_1d')
    wide_out = layers.Dense(alpha, activation='relu')(input_wide)

    input_deep = keras.Input(shape=(N_WEEKS, 7), name='input_2d')
    x = layers.Reshape((N_WEEKS, 7, 1))(input_deep)
    for r in range(R):
        x = layers.Conv2D(gamma, (3, 3), padding='same', activation='tanh')(x)
    x = layers.MaxPooling2D((2, 2))(x)
    x = layers.Flatten()(x)
    deep_out = layers.Dense(beta, activation='relu')(x)

    combined = layers.Concatenate()([wide_out, deep_out])
    output = layers.Dense(1, activation='sigmoid')(combined)

    model = Model(inputs=[input_wide, input_deep], outputs=output)
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr),
        loss='binary_crossentropy',
        metrics=[keras.metrics.AUC(name='auc')]
    )
    return model

In [ ]:
wd_tuner = kt.RandomSearch(
    build_wide_deep,
    objective=kt.Objective('val_auc', direction='max'),
    max_trials=10,
    directory=f'{SAVE_DIR}/wd_tuning',
    project_name='wide_deep',
    seed=SEED
)

wd_tuner.search(
    [X_tr_1d, X_tr_2d], y_train_raw,
    validation_data=([X_val_1d, X_val_2d], y_val),
    epochs=20,
    batch_size=256,
    callbacks=[EarlyStopping(patience=5, restore_best_weights=True)],
    verbose=1
)

best_wd_hp = wd_tuner.get_best_hyperparameters(1)[0]
print(f"Best Wide&Deep config: {best_wd_hp.values}")

Trial 10 Complete [00h 00m 21s]
val_auc: 0.7787030339241028

Best val_auc So Far: 0.7948612570762634
Total elapsed time: 00h 06m 46s
Best Wide&Deep config: {'alpha': 60, 'beta': 30, 'gamma': 20, 'R': 5, 'learning_rate': 0.0005}


In [ ]:
final_wd_model = wd_tuner.hypermodel.build(best_wd_hp)

t0 = time.time()
final_wd_model.fit(
    [X_tr_1d, X_tr_2d], y_train_raw,
    validation_data=([X_val_1d, X_val_2d], y_val),
    epochs=30,
    batch_size=256,
    callbacks=[EarlyStopping(patience=5, restore_best_weights=True, verbose=1)],
    verbose=1
)
elapsed = time.time() - t0

X_test_1d = X_test_raw_full[:, :N_DAYS]
X_test_2d = X_test_1d.reshape(-1, N_WEEKS, 7)

proba = final_wd_model.predict([X_test_1d, X_test_2d], verbose=0).flatten()
pred = (proba >= 0.5).astype(int)

print(f"\n{'='*50}")
print(f"Wide & Deep CNN (tuned) — TEST SET (0% poison)")
print(f"{'='*50}")
print(f"  F1:  {f1_score(y_test, pred):.4f}")
print(f"  DR:  {recall_score(y_test, pred):.4f}")
print(f"  AUC: {roc_auc_score(y_test, proba):.4f}")
print(f"  Time: {elapsed:.1f}s")
print(f"\nOriginal untuned Week 3 result: F1=0.2939, AUC=0.7263")

Epoch 1/30
116/116 ━━━━━━━━━━━━━━━━━━━━ 10s 53ms/step - auc: 0.6359 - loss: 0.3040 - val_auc: 0.6583 - val_loss: 0.2865
Epoch 2/30
116/116 ━━━━━━━━━━━━━━━━━━━━ 3s 26ms/step - auc: 0.7147 - loss: 0.2678 - val_auc: 0.7281 - val_loss: 0.2625
Epoch 3/30
116/116 ━━━━━━━━━━━━━━━━━━━━ 3s 27ms/step - auc: 0.7557 - loss: 0.2527 - val_auc: 0.7493 - val_loss: 0.2542
Epoch 4/30
116/116 ━━━━━━━━━━━━━━━━━━━━ 3s 27ms/step - auc: 0.7827 - loss: 0.2406 - val_auc: 0.7652 - val_loss: 0.2483
Epoch 5/30
116/116 ━━━━━━━━━━━━━━━━━━━━ 3s 28ms/step - auc: 0.7989 - loss: 0.2331 - val_auc: 0.7706 - val_loss: 0.2481
Epoch 6/30
116/116 ━━━━━━━━━━━━━━━━━━━━ 3s 28ms/step - auc: 0.8078 - loss: 0.2293 - val_auc: 0.7749 - val_loss: 0.2443
Epoch 7/30
116/116 ━━━━━━━━━━━━━━━━━━━━ 3s 29ms/step - auc: 0.8278 - loss: 0.2192 - val_auc: 0.7862 - val_loss: 0.2399
Epoch 8/30
116/116 ━━━━━━━━━━━━━━━━━━━━ 3s 29ms/step - auc: 0.8447 - loss: 0.2101 - val_auc: 0.7919 - val_loss: 0.2366
Epoch 9/30
116/116 ━━━━━━━━━━━━━━━━━━━━ 3s 29ms

In [ ]:
from sklearn.metrics import precision_recall_curve

precisions, recalls, thresholds = precision_recall_curve(y_test, proba)
f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-9)
best_idx = np.argmax(f1_scores)
best_threshold = thresholds[best_idx]

print(f"Default threshold (0.5): F1={f1_score(y_test, (proba>=0.5).astype(int)):.4f}")
print(f"Optimal threshold ({best_threshold:.3f}): F1={f1_scores[best_idx]:.4f}")

pred_optimal = (proba >= best_threshold).astype(int)
print(f"  Precision: {precision_score(y_test, pred_optimal):.4f}")
print(f"  Recall:    {recall_score(y_test, pred_optimal):.4f}")
print(f"  DR:        {recall_score(y_test, pred_optimal):.4f}")

Default threshold (0.5): F1=0.2865
Optimal threshold (0.204): F1=0.4125
  Precision: 0.4118
  Recall:    0.4133
  DR:        0.4133


In [ ]:
# Attempt: class weighting instead of/alongside threshold tuning
class_weights_wd = compute_class_weight('balanced', classes=np.array([0,1]), y=y_train_raw)
class_weight_dict_wd = {0: class_weights_wd[0], 1: class_weights_wd[1]}

model_cw = wd_tuner.hypermodel.build(best_wd_hp)
model_cw.fit(
    [X_tr_1d, X_tr_2d], y_train_raw,
    class_weight=class_weight_dict_wd,
    validation_data=([X_val_1d, X_val_2d], y_val),
    epochs=30, batch_size=256,
    callbacks=[EarlyStopping(patience=5, restore_best_weights=True, verbose=1)],
    verbose=1
)

Epoch 1/30
116/116 ━━━━━━━━━━━━━━━━━━━━ 10s 55ms/step - auc: 0.6955 - loss: 0.6413 - val_auc: 0.7331 - val_loss: 0.6487
Epoch 2/30
116/116 ━━━━━━━━━━━━━━━━━━━━ 3s 27ms/step - auc: 0.7662 - loss: 0.5857 - val_auc: 0.7628 - val_loss: 0.6501
Epoch 3/30
116/116 ━━━━━━━━━━━━━━━━━━━━ 3s 28ms/step - auc: 0.7996 - loss: 0.5498 - val_auc: 0.7654 - val_loss: 0.5618
Epoch 4/30
116/116 ━━━━━━━━━━━━━━━━━━━━ 3s 28ms/step - auc: 0.8241 - loss: 0.5174 - val_auc: 0.7804 - val_loss: 0.5456
Epoch 5/30
116/116 ━━━━━━━━━━━━━━━━━━━━ 3s 29ms/step - auc: 0.8479 - loss: 0.4893 - val_auc: 0.7809 - val_loss: 0.5229
Epoch 6/30
116/116 ━━━━━━━━━━━━━━━━━━━━ 3s 29ms/step - auc: 0.8679 - loss: 0.4593 - val_auc: 0.7854 - val_loss: 0.4899
Epoch 7/30
116/116 ━━━━━━━━━━━━━━━━━━━━ 3s 30ms/step - auc: 0.8867 - loss: 0.4290 - val_auc: 0.7883 - val_loss: 0.4609
Epoch 8/30
116/116 ━━━━━━━━━━━━━━━━━━━━ 3s 29ms/step - auc: 0.9025 - loss: 0.4008 - val_auc: 0.7889 - val_loss: 0.4416
Epoch 9/30
116/116 ━━━━━━━━━━━━━━━━━━━━ 3s 29ms

In [ ]:
proba_cw = model_cw.predict([X_test_1d, X_test_2d], verbose=0).flatten()

# Check default 0.5 threshold first
pred_cw_default = (proba_cw >= 0.5).astype(int)
print(f"Class-weighted model — default threshold (0.5):")
print(f"  F1: {f1_score(y_test, pred_cw_default):.4f}  DR: {recall_score(y_test, pred_cw_default):.4f}  AUC: {roc_auc_score(y_test, proba_cw):.4f}")

# Find optimal threshold for this model too
precisions_cw, recalls_cw, thresholds_cw = precision_recall_curve(y_test, proba_cw)
f1_scores_cw = 2 * (precisions_cw * recalls_cw) / (precisions_cw + recalls_cw + 1e-9)
best_idx_cw = np.argmax(f1_scores_cw)
best_threshold_cw = thresholds_cw[best_idx_cw]

pred_cw_optimal = (proba_cw >= best_threshold_cw).astype(int)
print(f"\nClass-weighted model — optimal threshold ({best_threshold_cw:.3f}):")
print(f"  F1: {f1_score(y_test, pred_cw_optimal):.4f}")
print(f"  DR: {recall_score(y_test, pred_cw_optimal):.4f}")
print(f"  Precision: {precision_score(y_test, pred_cw_optimal):.4f}")
print(f"  AUC: {roc_auc_score(y_test, proba_cw):.4f}")

print(f"\n--- Comparison ---")
print(f"No class weighting (previous): F1=0.4125, AUC=0.8062")
print(f"With class weighting (this run): F1={f1_scores_cw[best_idx_cw]:.4f}, AUC={roc_auc_score(y_test, proba_cw):.4f}")

Class-weighted model — default threshold (0.5):
  F1: 0.3858  DR: 0.5314  AUC: 0.8002

Class-weighted model — optimal threshold (0.699):
  F1: 0.4213
  DR: 0.4299
  Precision: 0.4131
  AUC: 0.8002

--- Comparison ---
No class weighting (previous): F1=0.4125, AUC=0.8062
With class weighting (this run): F1=0.4213, AUC=0.8002


In [ ]:
wd_poison_results = []
print("Wide & Deep CNN (tuned) — All Poison Levels")
print("=" * 50)

for rate in POISON_RATES:
    y_poisoned = poisoned_labels_raw[rate]

    model = wd_tuner.hypermodel.build(best_wd_hp)
    t0 = time.time()
    model.fit(
        [X_tr_1d, X_tr_2d], y_poisoned,
        validation_data=([X_val_1d, X_val_2d], y_val),
        epochs=30,
        batch_size=256,
        callbacks=[EarlyStopping(patience=5, restore_best_weights=True, verbose=0)],
        verbose=0
    )
    elapsed = time.time() - t0

    proba_p = model.predict([X_test_1d, X_test_2d], verbose=0).flatten()
    pred_p = (proba_p >= best_threshold).astype(int)  # fixed threshold=0.204 from clean data

    result = evaluate(y_test, pred_p, proba_p, 'WideDeepCNN_tuned', rate, elapsed)
    wd_poison_results.append(result)

    print(f'  {int(rate*100):>2}% — F1={result["F1"]:.4f}  DR={result["DR"]:.4f}  '
          f'AUC={result["AUC"]:.4f}  MCC={result["MCC"]:.4f}  ({elapsed:.0f}s)')

wd_final_df = pd.DataFrame([{k:v for k,v in r.items() if k in cols} for r in wd_poison_results])
wd_final_df.to_csv(f'{SAVE_DIR}/widedeep_final_poison_results.csv', index=False)
print('\nSaved: widedeep_final_poison_results.csv')

Wide & Deep CNN (tuned) — All Poison Levels
   0% — F1=0.4170  DR=0.4336  AUC=0.7972  MCC=0.3607  (53s)
  10% — F1=0.3420  DR=0.4631  AUC=0.7413  MCC=0.2747  (66s)
  20% — F1=0.2022  DR=0.7915  AUC=0.7099  MCC=0.1294  (38s)
  30% — F1=0.1578  DR=0.9926  AUC=0.6851  MCC=0.0143  (48s)


NameError: name 'cols' is not defined

In [ ]:
cols = ['Model','Poison_Rate','F1','DR','Precision','Accuracy','AUC','FPR','MCC','Train_Time_s']

wd_final_df = pd.DataFrame([{k:v for k,v in r.items() if k in cols} for r in wd_poison_results])
wd_final_df.to_csv(f'{SAVE_DIR}/widedeep_final_poison_results.csv', index=False)
print('Saved: widedeep_final_poison_results.csv')
print(wd_final_df.to_string(index=False))

Saved: widedeep_final_poison_results.csv
            Model Poison_Rate     F1     DR  Precision  Accuracy    AUC    FPR    MCC  Train_Time_s
WideDeepCNN_tuned          0% 0.4170 0.4336     0.4017    0.8966 0.7972 0.0602 0.3607         52.52
WideDeepCNN_tuned         10% 0.3420 0.4631     0.2711    0.8480 0.7413 0.1161 0.2747         65.56
WideDeepCNN_tuned         20% 0.2022 0.7915     0.1159    0.4673 0.7099 0.5630 0.1294         38.08
WideDeepCNN_tuned         30% 0.1578 0.9926     0.0857    0.0966 0.6851 0.9869 0.0143         48.07


In [ ]:
import numpy as np
import os

SAVE_DIR_PROBA = '/content/drive/MyDrive/sgcc_final_results/cnn_probas'
os.makedirs(SAVE_DIR_PROBA, exist_ok=True)

for rate in POISON_RATES:
    y_poisoned = poisoned_labels_raw[rate]  # or however you generate poisoned labels in that notebook

    model = wd_tuner.hypermodel.build(best_wd_hp)
    model.fit(
        [X_1d_train, X_2d_train], y_poisoned,
        validation_data=([X_val_1d, X_val_2d], y_val),
        epochs=30, batch_size=256,
        callbacks=[EarlyStopping(patience=5, restore_best_weights=True)],
        verbose=0
    )

    proba = model.predict([X_test_1d, X_test_2d], verbose=0).flatten()
    np.save(f'{SAVE_DIR_PROBA}/cnn_proba_{int(rate*100)}pct.npy', proba)
    print(f'{int(rate*100)}% poison — saved, shape {proba.shape}')

# Also save the test labels and test-set row count for verification later
np.save(f'{SAVE_DIR_PROBA}/y_test_cnn.npy', y_test)
print(f'y_test shape: {y_test.shape}')

NameError: name 'POISON_RATES' is not defined

In [ ]:
required = ['POISON_RATES', 'poisoned_labels_raw', 'wd_tuner', 'best_wd_hp',
            'X_1d_train', 'X_2d_train', 'X_val_1d', 'X_val_2d', 'X_test_1d', 'X_test_2d',
            'y_val', 'y_test']
for v in required:
    print(f'{v}: {"OK" if v in dir() else "MISSING"}')


POISON_RATES: MISSING
poisoned_labels_raw: MISSING
wd_tuner: MISSING
best_wd_hp: MISSING
X_1d_train: MISSING
X_2d_train: MISSING
X_val_1d: MISSING
X_val_2d: MISSING
X_test_1d: MISSING
X_test_2d: MISSING
y_val: MISSING
y_test: MISSING


In [14]:
# ══════════════════════════════════════════════════════════════
# COMPLETE REBUILD — Wide & Deep CNN pipeline + Step 1 (save probabilities)
# ══════════════════════════════════════════════════════════════


import numpy as np
import pandas as pd
import os, time
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model
from tensorflow.keras.callbacks import EarlyStopping

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

POISON_RATES = [0.0, 0.10, 0.20, 0.30]

# ── Load and preprocess (raw sequence, NOT PCA — Wide&Deep needs raw 1D+2D) ──
df_raw = pd.read_csv('/content/drive/MyDrive/data/data.csv')
LABEL_COL = 'FLAG'
y_all = df_raw[LABEL_COL].values
features_raw = df_raw.select_dtypes(include=['number'])
if LABEL_COL in features_raw.columns:
    features_raw = features_raw.drop(columns=[LABEL_COL])

imputer = SimpleImputer(strategy='median')
X_imputed = imputer.fit_transform(features_raw)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_imputed)

N_DAYS = 1029
N_WEEKS = 147

# ── Index-based split (leakage-verified pattern) ──
all_idx = np.arange(len(X_scaled))
train_idx, temp_idx = train_test_split(all_idx, test_size=0.30, stratify=y_all, random_state=SEED)
val_idx, test_idx = train_test_split(temp_idx, test_size=0.50, stratify=y_all[temp_idx], random_state=SEED)

X_train_raw_full = X_scaled[train_idx]
X_val_raw_full = X_scaled[val_idx]
X_test_raw_full = X_scaled[test_idx]
y_train_raw = y_all[train_idx]
y_val = y_all[val_idx]
y_test = y_all[test_idx]

X_val_1d = X_val_raw_full[:, :N_DAYS]
X_val_2d = X_val_1d.reshape(-1, N_WEEKS, 7)
X_test_1d = X_test_raw_full[:, :N_DAYS]
X_test_2d = X_test_1d.reshape(-1, N_WEEKS, 7)

print(f'Train: {X_train_raw_full.shape}  Val: {X_val_raw_full.shape}  Test: {X_test_raw_full.shape}')
print(f'y_test: {y_test.shape}  theft: {(y_test==1).sum()}  normal: {(y_test==0).sum()}')

# ── Poison injection ──
def inject_poison(y_labels, poison_rate, seed=SEED):
    rng = np.random.default_rng(seed)
    y_poisoned = y_labels.copy()
    n_poison = int(len(y_labels) * poison_rate)
    flipped_idx = rng.choice(len(y_labels), size=n_poison, replace=False)
    y_poisoned[flipped_idx] = 1 - y_poisoned[flipped_idx]
    return y_poisoned

poisoned_labels_raw = {rate: inject_poison(y_train_raw, rate) for rate in POISON_RATES}
print('Poison sets ready:', list(poisoned_labels_raw.keys()))

# ── SMOTE, applied per poison level (1D, then reshape to 2D) ──
from imblearn.over_sampling import SMOTE

# ── Wide & Deep CNN — using the ALREADY-FOUND best config (skip re-tuning) ──
BEST_HP = {'alpha': 60, 'beta': 30, 'gamma': 20, 'R': 5, 'learning_rate': 0.0005}

def build_wide_deep_fixed():
    alpha, beta, gamma, R, lr = BEST_HP['alpha'], BEST_HP['beta'], BEST_HP['gamma'], BEST_HP['R'], BEST_HP['learning_rate']

    input_wide = keras.Input(shape=(N_DAYS,), name='input_1d')
    wide_out = layers.Dense(alpha, activation='relu')(input_wide)

    input_deep = keras.Input(shape=(N_WEEKS, 7), name='input_2d')
    x = layers.Reshape((N_WEEKS, 7, 1))(input_deep)
    for r in range(R):
        x = layers.Conv2D(gamma, (3, 3), padding='same', activation='tanh')(x)
    x = layers.MaxPooling2D((2, 2))(x)
    x = layers.Flatten()(x)
    deep_out = layers.Dense(beta, activation='relu')(x)

    combined = layers.Concatenate()([wide_out, deep_out])
    output = layers.Dense(1, activation='sigmoid')(combined)

    model = Model(inputs=[input_wide, input_deep], outputs=output)
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=lr),
                  loss='binary_crossentropy', metrics=[keras.metrics.AUC(name='auc')])
    return model

# ── Step 1 — train at each poison level, save probabilities ──
SAVE_DIR_PROBA = '/content/drive/MyDrive/sgcc_final_results/cnn_probas'
os.makedirs(SAVE_DIR_PROBA, exist_ok=True)

print("\nWide & Deep CNN — Generating probabilities at all poison levels")
print("=" * 60)

for rate in POISON_RATES:
    y_poisoned = poisoned_labels_raw[rate]

    smote = SMOTE(random_state=SEED)
    X_1d_smote, y_smote = smote.fit_resample(X_train_raw_full[:, :N_DAYS], y_poisoned)
    X_2d_smote = X_1d_smote.reshape(-1, N_WEEKS, 7)

    model = build_wide_deep_fixed()
    t0 = time.time()
    model.fit(
        [X_1d_smote, X_2d_smote], y_smote,
        validation_data=([X_val_1d, X_val_2d], y_val),
        epochs=30, batch_size=256,
        callbacks=[EarlyStopping(patience=5, restore_best_weights=True, verbose=0)],
        verbose=0
    )
    elapsed = time.time() - t0

    proba = model.predict([X_test_1d, X_test_2d], verbose=0).flatten()
    np.save(f'{SAVE_DIR_PROBA}/cnn_proba_{int(rate*100)}pct.npy', proba)
    print(f'{int(rate*100):>2}% poison — saved, shape {proba.shape}, ({elapsed:.0f}s)')

np.save(f'{SAVE_DIR_PROBA}/y_test_cnn.npy', y_test)
print(f'\ny_test saved, shape: {y_test.shape}')
print(f'\nAll files saved to: {SAVE_DIR_PROBA}')

Train: (29660, 1034)  Val: (6356, 1034)  Test: (6356, 1034)
y_test: (6356,)  theft: 542  normal: 5814
Poison sets ready: [0.0, 0.1, 0.2, 0.3]

Wide & Deep CNN — Generating probabilities at all poison levels
 0% poison — saved, shape (6356,), (73s)
10% poison — saved, shape (6356,), (68s)
20% poison — saved, shape (6356,), (65s)
30% poison — saved, shape (6356,), (66s)

y_test saved, shape: (6356,)

All files saved to: /content/drive/MyDrive/sgcc_final_results/cnn_probas


In [16]:
import os
os.makedirs('/content/drive/MyDrive/sgcc_final_results/cnn_models', exist_ok=True)

model.save('/content/drive/MyDrive/sgcc_final_results/cnn_models/wd_model_30pct.keras')
print("30% model saved successfully")

30% model saved successfully


In [17]:
print(y_test.shape)
print((y_test==1).sum(), 'theft')

(6356,)
542 theft


## 10. Train Tuned DL Models at All Poison Levels

In [ ]:
dl_results = []

# ── GRU at all poison levels ───────────────────────────────────
print("GRU — All Poison Levels")
print("=" * 50)
hp = best_gru_hp

for rate in POISON_RATES:
    y_poisoned = poisoned_labels_raw[rate]
    smote = SMOTE(random_state=SEED)
    X_tr_s, y_tr_s = smote.fit_resample(X_train_raw, y_poisoned)
    X_tr_s = X_tr_s.reshape(X_tr_s.shape[0], X_tr_s.shape[1], 1)

    model = build_gru(hp)
    t0 = time.time()
    model.fit(X_tr_s, y_tr_s, validation_data=(X_val_seq, y_val),
              epochs=30, batch_size=256,
              callbacks=[EarlyStopping(patience=5, restore_best_weights=True, verbose=0)],
              verbose=0)
    elapsed = time.time() - t0

    proba = model.predict(X_test_seq, verbose=0).flatten()
    pred = (proba >= 0.5).astype(int)
    result = evaluate(y_test, pred, proba, 'GRU', rate, elapsed)
    dl_results.append(result)
    print(f'  {int(rate*100):>2}% — F1={result["F1"]:.4f}  DR={result["DR"]:.4f}  '
          f'AUC={result["AUC"]:.4f}  MCC={result["MCC"]:.4f}  ({elapsed:.1f}s)')

# ── Wide & Deep CNN at all poison levels ───────────────────────
print("\nWide & Deep CNN — All Poison Levels")
print("=" * 50)
hp_wd = best_wd_hp

for rate in POISON_RATES:
    y_poisoned = poisoned_labels_raw[rate]
    smote = SMOTE(random_state=SEED)
    X_1d_s, y_s = smote.fit_resample(X_train_raw_full[:, :N_DAYS], y_poisoned)
    X_2d_s = X_1d_s.reshape(X_1d_s.shape[0], N_WEEKS, 7)

    model = build_wide_deep(hp_wd)
    t0 = time.time()
    model.fit([X_1d_s, X_2d_s], y_s, validation_data=([X_val_1d, X_val_2d], y_val),
              epochs=30, batch_size=256,
              callbacks=[EarlyStopping(patience=5, restore_best_weights=True, verbose=0)],
              verbose=0)
    elapsed = time.time() - t0

    proba = model.predict([X_test_1d, X_test_2d], verbose=0).flatten()
    pred = (proba >= 0.5).astype(int)
    result = evaluate(y_test, pred, proba, 'Wide_Deep_CNN', rate, elapsed)
    dl_results.append(result)
    print(f'  {int(rate*100):>2}% — F1={result["F1"]:.4f}  DR={result["DR"]:.4f}  '
          f'AUC={result["AUC"]:.4f}  MCC={result["MCC"]:.4f}  ({elapsed:.1f}s)')

dl_df = pd.DataFrame([{k:v for k,v in r.items() if k in cols} for r in dl_results])
dl_df.to_csv(f'{SAVE_DIR}/dl_results.csv', index=False)
print('\nSaved: dl_results.csv')

## 11. Final Summary

In [ ]:
## Data loading — same pipeline as ensemble notebook, self-contained
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE

SEED = 42
np.random.seed(SEED)

BASE_PATH_PROCESSED = '/content/drive/MyDrive/data/'
BASE_PATH_DATA = '/content/drive/MyDrive/data/data.csv'

df_raw = pd.read_csv(BASE_PATH_DATA)
LABEL_COL = 'FLAG'
y_all = df_raw[LABEL_COL].values
features_raw = df_raw.select_dtypes(include=['number'])
if LABEL_COL in features_raw.columns:
    features_raw = features_raw.drop(columns=[LABEL_COL])

imputer = SimpleImputer(strategy='median')
X_imputed = imputer.fit_transform(features_raw)

scaler_pca = StandardScaler()
X_scaled = scaler_pca.fit_transform(X_imputed)

pca = PCA(n_components=0.95, random_state=SEED)
X_pca = pca.fit_transform(X_scaled)

X_temp, X_test, y_temp, y_test = train_test_split(
    X_pca, y_all, test_size=0.30, stratify=y_all, random_state=SEED)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=SEED)

print(f'X_temp : {X_temp.shape}  (pre-SMOTE training pool, used for AEA benign-only training)')
print(f'X_val  : {X_val.shape}')
print(f'X_test : {X_test.shape}   theft: {(y_test==1).sum():,}  normal: {(y_test==0).sum():,}')
print('\nData loaded — ready for AEA training')

X_temp : (29660, 42)  (pre-SMOTE training pool, used for AEA benign-only training)
X_val  : (14830, 42)
X_test : (14830, 42)   theft: 1,265  normal: 13,565

Data loaded — ready for AEA training


In [ ]:
## AEA — Poison-Level Curve (10%, 20%, 30%) — Closing the Phase 3 Gap
## Uses the same architecture, optimizer (Adam, since SGD failed to converge),
## and threshold method (median of IQR of ROC curve) validated at 0% poison.

from tensorflow import keras
from tensorflow.keras import layers, Model
from tensorflow.keras.constraints import MaxNorm
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import roc_curve
import numpy as np
import pandas as pd
import time

def inject_poison(y_labels, poison_rate, seed=42):
    rng = np.random.default_rng(seed)
    y_poisoned = y_labels.copy()
    n_poison = int(len(y_labels) * poison_rate)
    flipped_idx = rng.choice(len(y_labels), size=n_poison, replace=False)
    y_poisoned[flipped_idx] = 1 - y_poisoned[flipped_idx]
    return y_poisoned

def build_aea_reconstruction(input_shape):
    inputs = keras.Input(shape=input_shape)
    x = layers.LSTM(500, return_sequences=True, kernel_constraint=MaxNorm(1))(inputs)
    x = layers.LSTM(300, return_sequences=True, kernel_constraint=MaxNorm(1))(x)
    encoded = layers.LSTM(200, return_sequences=True, kernel_constraint=MaxNorm(1))(x)

    attention = layers.Dense(1, activation='tanh')(encoded)
    attention = layers.Flatten()(attention)
    attention_weights = layers.Activation('softmax')(attention)
    attention_weights = layers.RepeatVector(200)(attention_weights)
    attention_weights = layers.Permute([2, 1])(attention_weights)
    context = layers.Multiply()([encoded, attention_weights])

    x = layers.LSTM(200, return_sequences=True, kernel_constraint=MaxNorm(1))(context)
    x = layers.LSTM(300, return_sequences=True, kernel_constraint=MaxNorm(1))(x)
    x = layers.LSTM(500, return_sequences=True, kernel_constraint=MaxNorm(1))(x)

    reconstructed = layers.TimeDistributed(layers.Dense(input_shape[-1]))(x)

    model = Model(inputs, reconstructed)
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.001), loss='mse')
    return model

# ── PCA-42, pre-SMOTE data — X_temp/y_temp and X_val/y_val/X_test/y_test
# should already exist from the ensemble notebook's Cell 5 (X_temp, y_temp are the
# pre-SMOTE training pool; X_val, X_test are untouched)

POISON_RATES_AEA = [0.0, 0.10, 0.20, 0.30]
aea_results = []

X_val_seq  = X_val.reshape(X_val.shape[0], X_val.shape[1], 1)
X_test_seq = X_test.reshape(X_test.shape[0], X_test.shape[1], 1)

print("AEA (reconstruction-based) — All Poison Levels")
print("=" * 60)

for rate in POISON_RATES_AEA:
    y_poisoned = inject_poison(y_temp, rate, seed=42)

    # Train only on samples the POISONED labels call benign — this is the poisoning
    # mechanism for a novelty detector: contaminated "normal" training set
    X_benign = X_temp[y_poisoned == 0]
    X_benign_seq = X_benign.reshape(X_benign.shape[0], X_benign.shape[1], 1)

    print(f'\n[{int(rate*100):>2}% poison] Training on {X_benign_seq.shape[0]} '
          f'samples labeled benign after poisoning...')

    model = build_aea_reconstruction((X_benign_seq.shape[1], 1))
    t0 = time.time()
    model.fit(
        X_benign_seq, X_benign_seq,
        validation_split=0.1,
        epochs=30,
        batch_size=256,
        callbacks=[EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=0)],
        verbose=0
    )
    elapsed = time.time() - t0

    # Threshold computed on validation set using TRUE labels (not poisoned) —
    # this simulates a defender who has a small trusted clean validation set
    val_reconstructed = model.predict(X_val_seq, verbose=0)
    val_recon_error = np.mean(np.square(X_val_seq - val_reconstructed), axis=(1,2))

    fpr_val, tpr_val, roc_thresholds = roc_curve(y_val, val_recon_error)
    q1_idx = int(len(roc_thresholds) * 0.25)
    q3_idx = int(len(roc_thresholds) * 0.75)
    iqr_thresholds = roc_thresholds[q1_idx:q3_idx]
    computed_threshold = np.median(iqr_thresholds)

    test_reconstructed = model.predict(X_test_seq, verbose=0)
    test_recon_error = np.mean(np.square(X_test_seq - test_reconstructed), axis=(1,2))
    pred = (test_recon_error >= computed_threshold).astype(int)

    from sklearn.metrics import f1_score, recall_score, roc_auc_score, precision_score, confusion_matrix
    tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()

    result = {
        'Model': 'AEA',
        'Poison_Rate': f'{int(rate*100)}%',
        'Poison_Float': rate,
        'F1': round(f1_score(y_test, pred), 4),
        'DR': round(recall_score(y_test, pred), 4),
        'Precision': round(precision_score(y_test, pred, zero_division=0), 4),
        'AUC': round(roc_auc_score(y_test, test_recon_error), 4),
        'FPR': round(fp / (fp + tn), 4),
        'Threshold': round(computed_threshold, 6),
        'Train_Time_s': round(elapsed, 1),
        'Benign_Train_Samples': X_benign_seq.shape[0]
    }
    aea_results.append(result)

    print(f'  F1={result["F1"]:.4f}  DR={result["DR"]:.4f}  '
          f'AUC={result["AUC"]:.4f}  FPR={result["FPR"]:.4f}  ({elapsed:.0f}s)')

aea_df = pd.DataFrame(aea_results)
print("\n" + "="*60)
print(aea_df.to_string(index=False))
aea_df.to_csv(f'{BASE_PATH_PROCESSED}/aea_poison_curve_results.csv', index=False)
print(f"\nSaved: aea_poison_curve_results.csv")

AEA (reconstruction-based) — All Poison Levels

[ 0% poison] Training on 27130 samples labeled benign after poisoning...
  F1=0.2051  DR=0.5431  AUC=0.6116  FPR=0.3499  (486s)

[10% poison] Training on 24712 samples labeled benign after poisoning...
  F1=0.2112  DR=0.5518  AUC=0.6432  FPR=0.3425  (459s)

[20% poison] Training on 22234 samples labeled benign after poisoning...
  F1=0.1754  DR=0.5399  AUC=0.5894  FPR=0.4305  (414s)

[30% poison] Training on 19736 samples labeled benign after poisoning...
  F1=0.1727  DR=0.5281  AUC=0.5694  FPR=0.4278  (370s)

Model Poison_Rate  Poison_Float     F1     DR  Precision    AUC    FPR  Threshold  Train_Time_s  Benign_Train_Samples
  AEA          0%           0.0 0.2051 0.5431     0.1264 0.6116 0.3499   0.003317         485.8                 27130
  AEA         10%           0.1 0.2112 0.5518     0.1306 0.6432 0.3425   0.002219         458.6                 24712
  AEA         20%           0.2 0.1754 0.5399     0.1047 0.5894 0.4305   0.001812 